# RAG Chatbot

Chúng ta sẽ xây dựng hệ thống Retrieval-Augmented Generation (RAG)

Chương trình sử dụng 3 thư viện chính: `pypdf`, `chromadb`, `ollama`.

## **Cài đặt Ollama**

In [2]:
!ollama pull vicuna:7b-v1.5-q5_1
!ollama pull bge-m3

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling d5e2e60f5a2f:   0% ▕                  ▏ 856 KB/5.1 GB                  pulling manifest 
pulling d5e2e60f5a2f:   0% ▕                  ▏ 2.5 MB/5.1 GB                  pulling manifest 
pulling d5e2e60f5a2f:   0% ▕                  ▏ 2.9 MB/5.1 GB                  pulling manifest 
pulling d5e2e60f5a2f:   0% ▕                  ▏ 4.3 MB/5.1 GB                  pulling manifest 
pulling d5e2e60f5a2f:   0% ▕                  ▏ 5.8 MB/5.1 GB                  pulling manifest 
pulling d5e2e60f5a2f:   0% ▕                  ▏ 6.5 MB/5.1 GB                  

## **Download file**

In [3]:
!gdown 1lWuq0COKnU9mCfMvTEq54DBLgAh3yYDx

'gdown' is not recognized as an internal or external command,
operable program or batch file.


## **Install & import**

In [4]:
!pip install -q pypdf chromadb ollama

In [5]:
import pypdf
import chromadb
import ollama

## **1. Đọc PDF**

In [6]:
reader = pypdf.PdfReader("[2026] LeWorldModel Stable End-to-End JAPE from Pixels.pdf")
full_text = "\n".join(page.extract_text() or "" for page in reader.pages)
print("Số trang:", len(reader.pages))
print("Tổng ký tự:", len(full_text))

Số trang: 28
Tổng ký tự: 81251


## **2. Fixed-size chunking**

In [7]:
def chunk_text(text, size=1000, overlap=200):
    paras = [p.strip() for p in text.split("\n") if p.strip()]
    chunks, cur = [], ""
    for p in paras:
        # Nếu một đoạn dài hơn size, cắt nhỏ đoạn đó (vẫn giữ overlap)
        while len(p) > size:
            if cur:
                chunks.append(cur.strip())
                cur = ""
            chunks.append(p[:size].strip())
            p = p[size - overlap:]
        if len(cur) + len(p) + 1 <= size:
            cur += p + "\n"
        else:
            if cur:
                chunks.append(cur.strip())
            cur = (cur[-overlap:] + p + "\n") if overlap else (p + "\n")
    if cur.strip():
        chunks.append(cur.strip())
    return chunks

chunks = chunk_text(full_text)
print("Số chunks:", len(chunks))
print(chunks[5][:300])

Số chunks: 107
d-free, with a single hyperparameter with provable anti-collapse guarantees.
1 Introduction
A central goal of artificial intelligence is to develop agents that acquire skills across diverse tasks and
environments using a single, unified learning par adigm—one that operates directly from sensory in-



## **3. Embedding + Vector store (Chroma native)**

In [8]:
def embed(texts):
    return ollama.embed(model="bge-m3", input=texts)["embeddings"]

client = chromadb.Client()
collection = client.get_or_create_collection("rag")

collection.add(
    ids=[str(i) for i in range(len(chunks))],
    documents=chunks,
    embeddings=embed(chunks),
)
print("Đã index:", collection.count(), "chunks")

Đã index: 107 chunks


In [9]:
!ollama list

NAME                   ID              SIZE      MODIFIED          
bge-m3:latest          790764642607    1.2 GB    About an hour ago    
vicuna:7b-v1.5-q5_1    fa2d15c41d43    5.1 GB    About an hour ago    


## **4. Retrieve**

In [11]:
def retrieve(query, k=4):
    res = collection.query(query_embeddings=embed([query]), n_results=k)
    return res["documents"][0]

QUERY = "LeWM kiến trúc khác DINO-WM ở điểm nào?"
for doc in retrieve(QUERY):
    print(doc[:200])
    print("-" * 40)

LeWM, which may be explained by
the SIGReg regularization encouraging a Gaussian distribution in a high-dimensional latent space, while the
intrinsic dimensionality of the environment is much lower.
D
----------------------------------------
oss architectures and hyperparameter choices, while enabling efficient logarithmic-
time hyperparameter search.
• Our experiment demonstrates that LeWM achieves competitive control performance across

----------------------------------------
on averaged
over 50 runs. Encoding observations with ∼200× fewer tokens than DINO-WM allows LeWM to achieve
planning speeds comparable to PLDM while being up to ∼50× faster than DINO-WM.Center–Right:

----------------------------------------
as feature encoder to mitigate representation collapse, but its original formulation additionally
incorporates other modalities, such as proprioceptive inputs; for a fair comparison, unless specified

----------------------------------------


## **5. LLM + RAG**

In [ ]:
PROMPT = """Bạn là trợ lý hỏi đáp. Dùng các đoạn ngữ cảnh dưới đây để trả lời câu hỏi.
Nếu ngữ cảnh không có thông tin, hãy nói là bạn không biết, đừng bịa.
Trả lời ngắn gọn, chính xác, bằng tiếng Việt.

Ngữ cảnh:
{context}

Câu hỏi: {question}

Trả lời:"""

def rag(question, k=4):
    context = "\n\n".join(retrieve(question, k))
    resp = ollama.chat(
        model="vicuna:7b-v1.5-q5_1",
        messages=[{"role": "user", "content": PROMPT.format(context=context, question=question)}],
        options={"temperature": 0},
    )
    return resp["message"]["content"]

print(rag("LeWM kiến trúc khác DINO-WM ở điểm nào?"))

YOLOv10 là một phiên bản của mô hình máy học được đề xuất trong bài báo "YOLOv10: Real-Time End-to-End Object Detection" của nhóm tác giả tại Trường Quốc gia Học viện (Tsinghua University). Mô hình này đã đạt được hiệu suất vượt trội hơn so với các phiên bản YOLO trước đó ở các khía cạnh khác nhau, tăng cường khả năng phát hiện đối tượng theo thời gian thực (real-time object detection). YOLOv10 sử dụng backbone mới EfficientRep và Rep-PAN Neck để tối ưu hóa và tăng hiệu năng của mô hình.


# Streamlit Deployment

In [12]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 112.1 MB/s eta 0:00:00


In [ ]:
%%writefile chatbot_app.py
import streamlit as st
import tempfile, os, time
import pypdf
import chromadb
import ollama

LLM_MODEL = "vicuna:7b-v1.5-q5_1"
EMBED_MODEL = "bge-m3"

PROMPT = """Bạn là trợ lý hỏi đáp. Dùng các đoạn ngữ cảnh dưới đây để trả lời câu hỏi.
Nếu ngữ cảnh không có thông tin, hãy nói là bạn không biết, đừng bịa.
Trả lời ngắn gọn, chính xác, bằng tiếng Việt.

Ngữ cảnh:
{context}

Câu hỏi: {question}

Trả lời:"""

for k, v in {"collection": None, "pdf_name": "", "chat_history": []}.items():
    st.session_state.setdefault(k, v)

def embed(texts):
    return ollama.embed(model=EMBED_MODEL, input=texts)["embeddings"]

def chunk_text(text, size=1000, overlap=200):
    paras = [p.strip() for p in text.split("\n") if p.strip()]
    chunks, cur = [], ""
    for p in paras:
        # Nếu một đoạn dài hơn size, cắt nhỏ đoạn đó (vẫn giữ overlap)
        while len(p) > size:
            if cur:
                chunks.append(cur.strip())
                cur = ""
            chunks.append(p[:size].strip())
            p = p[size - overlap:]
        if len(cur) + len(p) + 1 <= size:
            cur += p + "\n"
        else:
            if cur:
                chunks.append(cur.strip())
            cur = (cur[-overlap:] + p + "\n") if overlap else (p + "\n")
    if cur.strip():
        chunks.append(cur.strip())
    return chunks

chunks = chunk_text(full_text)
print("Số chunks:", len(chunks))
print(chunks[5][:300])

def process_pdf(uploaded_file):
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(uploaded_file.getvalue())
        path = tmp.name
    text = "\n".join(p.extract_text() or "" for p in pypdf.PdfReader(path).pages)
    os.unlink(path)

    chunks = chunk_text(text)
    client = chromadb.Client()
    col = client.get_or_create_collection(f"rag_{int(time.time())}")
    col.add(ids=[str(i) for i in range(len(chunks))], documents=chunks, embeddings=embed(chunks))
    return col, len(chunks)

def rag(question, collection, k=4):
    res = collection.query(query_embeddings=embed([question]), n_results=k)
    context = "\n\n".join(res["documents"][0])
    resp = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": PROMPT.format(context=context, question=question)}],
        options={"temperature": 0},
    )
    return resp["message"]["content"]

st.set_page_config(page_title="PDF RAG Chatbot", layout="wide", initial_sidebar_state="expanded")
st.title("PDF RAG Assistant")

with st.sidebar:
    st.subheader("📄 Upload tài liệu")
    f = st.file_uploader("Chọn file PDF", type="pdf")
    if f and st.button("🔄 Xử lý PDF", use_container_width=True):
        with st.spinner("Đang xử lý..."):
            st.session_state.collection, n = process_pdf(f)
            st.session_state.pdf_name = f.name
            st.session_state.chat_history = []
        st.success(f"✅ {n} chunks")
    st.info(f"📄 {st.session_state.pdf_name}" if st.session_state.pdf_name else "📄 Chưa có tài liệu")
    if st.button("🗑️ Xóa lịch sử chat", use_container_width=True):
        st.session_state.chat_history = []

for m in st.session_state.chat_history:
    with st.chat_message(m["role"]):
        st.write(m["content"])

if st.session_state.collection is None:
    st.info("🔄 Upload và xử lý PDF trước khi chat.")
    st.chat_input("Nhập câu hỏi...", disabled=True)
else:
    q = st.chat_input("Nhập câu hỏi của bạn...")
    if q:
        st.session_state.chat_history.append({"role": "user", "content": q})
        with st.chat_message("user"):
            st.write(q)
        with st.chat_message("assistant"):
            with st.spinner("Đang suy nghĩ..."):
                ans = rag(q, st.session_state.collection)
                st.write(ans)
        st.session_state.chat_history.append({"role": "assistant", "content": ans})

Overwriting chatbot_app.py


## **Triển khai**

In [15]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [20]:
import subprocess, time, requests
def ollama_up():
    try:
        return requests.get("http://localhost:11434", timeout=2).ok
    except Exception:
        return False
if not ollama_up():
    subprocess.Popen(["ollama", "serve"]); time.sleep(8)
print("Ollama running:", ollama_up())
!ollama list

Ollama running: True
NAME                   ID              SIZE      MODIFIED      
bge-m3:latest          790764642607    1.2 GB    8 minutes ago    
vicuna:7b-v1.5-q5_1    fa2d15c41d43    5.1 GB    8 minutes ago    


In [23]:
!streamlit run chatbot_app.py \
    --server.port 8501 --server.headless true \
    --server.enableCORS false --server.enableXsrfProtection false \
    &>/content/st.log &
import time; time.sleep(8)
print("Streamlit đang chạy ở cổng 8501")

Streamlit đang chạy ở cổng 8501


In [24]:
!./cloudflared tunnel --url http://localhost:8501

2026-06-18T02:32:37Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-18T02:32:37Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-18T02:32:41Z INF +--------------------------------------------------------------------------------------------+
2026-06-18T02:32:41Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-18T02:32:41Z INF |  https://others-able-gray-scenes.trycloudflare.com    